# 07 — Monitor progres dataset

Jalankan semua sel dengan kernel Python dari venv proyek. Notebook ini hanya membaca berkas lokal; tidak melakukan scraping, embedding, atau panggilan SerpApi/Gemini. Jalankan lagi setelah proses pengumpulan atau build untuk melihat angka terbaru.

**Arti angka:** URL kandidat adalah halaman dalam manifest; URL dicoba berarti ada checkpoint crawling; pasangan utama adalah satu query dan satu artikel Google Top 10; baris siap model berasal dari model_ready.csv. Satu artikel dapat menjadi pasangan untuk beberapa query. Sitasi Gemini di luar Top 10 dihitung sebagai dataset tambahan, bukan baris model utama.

In [1]:
import csv
import json
from collections import Counter
from datetime import datetime, timezone
from html import escape
from pathlib import Path
from IPython.display import HTML, display

csv.field_size_limit(100_000_000)
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data" / "raw" / "main").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari dalam folder proyek final-assignment.")
print("Proyek:", ROOT)

def read_json(path, default=None):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default

def read_csv(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def article_ids(path):
    if not path.exists():
        return set()
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return {row["article_id"] for row in csv.DictReader(handle)}

def yes(value):
    return value is True or value == "True"

def table(title, headers, rows):
    head = "".join("<th>" + escape(str(h)) + "</th>" for h in headers)
    body = "".join("<tr>" + "".join("<td>" + escape(str(v)) + "</td>" for v in row)
                   + "</tr>" for row in rows)
    display(HTML("<h3>" + escape(title) + "</h3>"
                 "<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
                 "<thead><tr>" + head + "</tr></thead><tbody>" + body
                 + "</tbody></table></div>"
                 "<style>th,td{border:1px solid #aaa;padding:5px 9px;text-align:left}"
                 "th{background:#eee}</style>"))

Proyek: d:\Kuliah\TA\final-assignment


In [2]:
BATCHES = {}
for query_dir in sorted((ROOT / "data/raw/main").glob("main_*")):
    batch = query_dir.name
    query_manifest = read_json(query_dir / "manifest.json")
    if not query_manifest:
        continue
    article_batch = "articles_" + batch
    raw_articles = ROOT / "data/raw/articles" / article_batch
    processed = ROOT / "data/processed" / article_batch
    article_manifest = read_json(raw_articles / "manifest.json", {"articles": [], "pairs": []})
    query_rows = {r["query_id"]: r for r in read_csv(ROOT / "data/interim/main" / batch / "queries.csv")}
    record_paths = list((raw_articles / "records").glob("*.json"))
    crawl_status = Counter()
    success_ids = set()
    for path in record_paths:
        record = read_json(path, {})
        status = record.get("status", "unknown")
        crawl_status[status] += 1
        if status == "success":
            success_ids.add(path.stem)
    summary = read_json(processed / "summary.json", {})
    build_time = summary.get("created_at")
    changed_after_build = 0
    if build_time:
        built_at = datetime.fromisoformat(build_time).timestamp()
        changed_after_build = sum(p.stat().st_mtime > built_at + 1 for p in record_paths)
    google_pairs = [p for p in article_manifest["pairs"] if yes(p.get("in_google_top10"))]
    BATCHES[batch] = {
        "queries": query_manifest["queries"],
        "query_rows": query_rows,
        "article_manifest": article_manifest,
        "google_pairs": google_pairs,
        "candidate_ids": {a["article_id"] for a in article_manifest["articles"]},
        "attempted_ids": {p.stem for p in record_paths},
        "success_ids": success_ids,
        "crawl_status": crawl_status,
        "built_article_ids": article_ids(processed / "articles.csv"),
        "primary": read_csv(processed / "dataset.csv"),
        "supplement": read_csv(processed / "dataset_gemini_only.csv"),
        "ready": read_csv(processed / "model_ready.csv"),
        "build_time": build_time or "belum ada",
        "changed_after_build": changed_after_build,
    }
print("Dibaca pada:", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Batch:", ", ".join(BATCHES) or "belum ada")

Dibaca pada: 2026-10-09T07:18:43+00:00
Batch: main_01, main_02, main_03


## Ringkasan keseluruhan

Angka URL di bawah memakai ID artikel unik lintas batch. Angka pasangan adalah jumlah baris, sehingga bisa lebih besar daripada jumlah URL. Data siap model dibaca dari hasil build terakhir; checkpoint crawling yang belum dibuild tidak otomatis muncul di sana.

In [3]:
all_queries = [q for b in BATCHES.values() for q in b["queries"]]
all_candidates = set().union(*(b["candidate_ids"] for b in BATCHES.values()))
all_attempted = set().union(*(b["attempted_ids"] for b in BATCHES.values()))
all_success = set().union(*(b["success_ids"] for b in BATCHES.values()))
all_primary = [r for b in BATCHES.values() for r in b["primary"]]
all_supplement = [r for b in BATCHES.values() for r in b["supplement"]]
all_ready = [r for b in BATCHES.values() for r in b["ready"]]
query_complete = sum(yes(b["query_rows"].get(q["query_id"], {}).get("collection_complete"))
                     for b in BATCHES.values() for q in b["queries"])
query_eligible = sum(yes(b["query_rows"].get(q["query_id"], {}).get("grounding_eligible"))
                     for b in BATCHES.values() for q in b["queries"])
pending_build = sum(len(b["attempted_ids"] - b["built_article_ids"])
                    for b in BATCHES.values())
overview = [
    ("Query terjadwal/accepted", len(all_queries)),
    ("Query selesai dikumpulkan", query_complete),
    ("Query belum lengkap", len(all_queries) - query_complete),
    ("Query memenuhi syarat grounding", query_eligible),
    ("URL kandidat unik, Google + Gemini", len(all_candidates)),
    ("URL unik sudah dicoba crawling", len(all_attempted)),
    ("URL unik berhasil diunduh", len(all_success)),
    ("URL kandidat belum dicoba", len(all_candidates - all_attempted)),
    ("Checkpoint dicoba tetapi belum masuk build", pending_build),
    ("Pasangan utama Google Top 10 dalam dataset.csv", len(all_primary)),
    ("Pasangan tambahan Gemini-only", len(all_supplement)),
    ("Pasangan siap model dalam model_ready.csv", len(all_ready)),
    ("Artikel unik dalam model_ready", len({r["article_id"] for r in all_ready})),
    ("Label 1 dalam model_ready", sum(r["citation_label"] == "1" for r in all_ready)),
    ("Label 0 dalam model_ready", sum(r["citation_label"] == "0" for r in all_ready)),
    ("Baris siap model tanpa usia publikasi", sum(not r.get("publication_age_days") for r in all_ready)),
]
table("Jumlah dataset saat ini", ["Ukuran", "Jumlah"], overview)

Ukuran,Jumlah
Query terjadwal/accepted,381
Query selesai dikumpulkan,370
Query belum lengkap,11
Query memenuhi syarat grounding,364
"URL kandidat unik, Google + Gemini",8275
URL unik sudah dicoba crawling,3796
URL unik berhasil diunduh,2946
URL kandidat belum dicoba,4479
Checkpoint dicoba tetapi belum masuk build,0
Pasangan utama Google Top 10 dalam dataset.csv,2869


## Progres query

“Selesai” mengikuti collection_complete pada ekspor query. “Memenuhi syarat” mengikuti grounding_eligible. Query yang belum lengkap bisa saja sudah pernah dicoba dan perlu dilanjutkan; jumlahnya bukan otomatis jumlah query PAA baru yang perlu dicari.

In [4]:
domains = ["kesehatan", "keuangan", "teknologi"]
query_table = []
ready_queries = {r["query_id"] for r in all_ready}
pending_query_rows = []
for domain in domains:
    scheduled = []
    complete = eligible = without_record = ready_count = 0
    for batch, b in BATCHES.items():
        for q in b["queries"]:
            if q["domain"] != domain:
                continue
            scheduled.append(q)
            r = b["query_rows"].get(q["query_id"], {})
            complete += yes(r.get("collection_complete"))
            eligible += yes(r.get("grounding_eligible"))
            without_record += not (ROOT / "data/raw/main" / batch / "queries"
                                   / (q["query_id"] + ".json")).exists()
            ready_count += q["query_id"] in ready_queries
            if not yes(r.get("collection_complete")):
                pending_query_rows.append((domain, batch, q["query_text"],
                                           r.get("google_status", "belum ada"),
                                           r.get("n_trial_records", "0"),
                                           r.get("n_valid", "0")))
    query_table.append((domain, len(scheduled), complete, eligible,
                        len(scheduled) - complete, without_record, ready_count))
table("Query per domain",
      ["Domain", "Terjadwal", "Selesai", "Grounding eligible",
       "Belum lengkap", "Belum pernah dimulai", "Punya baris model-ready"],
      query_table)
table("Query belum lengkap (maksimal 30 ditampilkan)",
      ["Domain", "Batch", "Query", "Status Google", "Percobaan tersimpan", "Valid"],
      pending_query_rows[:30])

Domain,Terjadwal,Selesai,Grounding eligible,Belum lengkap,Belum pernah dimulai,Punya baris model-ready
kesehatan,155,150,150,5,0,131
keuangan,131,128,124,3,0,107
teknologi,95,92,90,3,0,74


Domain,Batch,Query,Status Google,Percobaan tersimpan,Valid
kesehatan,main_02,Apa saja obat diare alami yang aman untuk ibu hamil?,error,0,0
kesehatan,main_02,Apakah cetirizine aman dikonsumsi oleh ibu menyusui?,error,0,0
kesehatan,main_03,Gejala awal penyakit jantung seperti apa?,completed,2,1
kesehatan,main_03,Apa arti SPF 50 PA+++?,error,0,0
kesehatan,main_03,Apa saja jenis obat yang berbahaya untuk ginjal?,error,0,0
keuangan,main_02,Berapa harga 1 gram emas hari ini?,started,0,0
keuangan,main_03,Berapa minimal gaji yang tidak kena pajak?,error,0,0
keuangan,main_03,Saldo mengendap BTN apakah bisa diambil?,completed,2,1
teknologi,main_02,Berapa harga laptop Asus dengan RAM 4GB dan Windows 10?,error,0,0
teknologi,main_03,Ctrl + F di Excel untuk apa?,completed,2,1


## Artikel dan pasangan per domain

“URL Google kandidat” dan “URL Google dicoba” menghitung artikel unik dalam Google Top 10. Satu artikel yang cocok dengan beberapa query hanya dihitung sekali pada dua kolom ini. “Pasangan utama” dan “baris model-ready” menghitung pasangan query–artikel. Artikel yang sama dapat muncul di lebih dari satu domain; karena itu jumlah artikel unik per domain tidak selalu dapat dijumlahkan sebagai angka unik keseluruhan.

In [5]:
article_table = []
for domain in domains:
    google_pairs = [p for b in BATCHES.values() for p in b["google_pairs"]
                    if p["domain"] == domain]
    candidate_urls = {p["article_id"] for p in google_pairs}
    attempted_urls = {p["article_id"] for b in BATCHES.values()
                      for p in b["google_pairs"]
                      if p["domain"] == domain and p["article_id"] in b["attempted_ids"]}
    success_urls = {p["article_id"] for b in BATCHES.values()
                    for p in b["google_pairs"]
                    if p["domain"] == domain and p["article_id"] in b["success_ids"]}
    primary = [r for r in all_primary if r["domain"] == domain]
    ready = [r for r in all_ready if r["domain"] == domain]
    article_table.append((domain, len(candidate_urls), len(attempted_urls),
                          len(success_urls), len(candidate_urls - attempted_urls),
                          len({r["article_id"] for r in primary}), len(primary),
                          len({r["article_id"] for r in ready}), len(ready),
                          sum(r["citation_label"] == "0" for r in ready),
                          sum(r["citation_label"] == "1" for r in ready)))
table("Google Top 10 dan data siap model per domain",
      ["Domain", "URL Google kandidat", "URL Google dicoba", "URL Google berhasil",
       "URL Google tersisa", "URL unik di dataset utama", "Pasangan utama",
       "Artikel unik siap model", "Baris siap model", "Label 0", "Label 1"],
      article_table)

Domain,URL Google kandidat,URL Google dicoba,URL Google berhasil,URL Google tersisa,URL unik di dataset utama,Pasangan utama,Artikel unik siap model,Baris siap model,Label 0,Label 1
kesehatan,1204,1125,833,79,1125,1186,500,542,422,120
keuangan,958,915,671,43,915,993,393,446,333,113
teknologi,718,661,478,57,661,690,210,217,161,56


## Sisa pekerjaan dan kebaruan hasil

Angka “URL Google tersisa” menunjukkan backlog scraping yang langsung dapat menambah dataset utama. URL Gemini-only yang tersisa memengaruhi dataset tambahan. Jika checkpoint lebih baru daripada build, jalankan kembali builder untuk memperbarui model_ready.csv.

In [6]:
(venv) PS D:\Kuliah\TA\final-assignment> python src/continue_main_dataset.py --run --max-queries 8 --check-every 5                                                       
Batch: main_02 | Query: 265 {'kesehatan': 115, 'keuangan': 92, 'teknologi': 58}
Pemantauan: {'checked_at': '2026-10-06T19:38:23.039624+00:00', 'gemini_usage_this_batch': {'query_records': 265, 'completed_queries': 261, 'trial_records': 783, 'error_trials': 3, 'archived_failed_attempts': 2, 'responses_with_usage': 780, 'promptTokenCount': 426770, 'candidatesTokenCount': 799017, 'totalTokenCount': 2511872, 'thoughtsTokenCount': 1286085, 'cachedContentTokenCount': 74565}, 'gemini_balance': 'not_available_from_generate_content_api', 'serpapi': {'checked_at': '2026-10-06T19:38:24.405098+00:00', 'total_searches_left': 131, 'plan_searches_left': 131, 'searches_per_month': 250, 'this_month_usage': 119}}
Pemantauan: {'checked_at': '2026-10-06T19:38:25.590545+00:00', 'gemini_usage_this_batch': {'query_records': 265, 'completed_queries': 261, 'trial_records': 783, 'error_trials': 3, 'archived_failed_attempts': 2, 'responses_with_usage': 780, 'promptTokenCount': 426770, 'candidatesTokenCount': 799017, 'totalTokenCount': 2511872, 'thoughtsTokenCount': 1286085, 'cachedContentTokenCount': 74565}, 'gemini_balance': 'not_available_from_generate_content_api', 'serpapi': {'checked_at': '2026-10-06T19:38:26.563313+00:00', 'total_searches_left': 131, 'plan_searches_left': 131, 'searches_per_month': 250, 'this_month_usage': 119}}
Checkpoint Google gagal/tidak pasti. Periksa sebelum mengulang; tidak ada retry API otomatis.

SyntaxError: invalid syntax (1621999956.py, line 1)

**Catatan:** “Siap model” berarti lolos pemeriksaan artikel, label, dan embedding yang diterapkan builder. Sebagian fitur seperti usia publikasi masih bisa kosong dan perlu ditangani saat pemodelan. Notebook ini tidak memutuskan kelayakan artikel baru dan tidak memperbarui dataset; untuk itu jalankan proses review dan build seperti biasa.